# NYC 311 2025 — Socioeconomic Analysis

This notebook integrates NYC 311 service requests with Census tract-level socioeconomic data to examine service request rates, resolution patterns, and observed associations across neighborhoods.

**Analysis period:** January–December 2025

Associations are descriptive and do not imply causality.


## 1. Load Analytical Dataset



In [1]:
# NOTEBOOK 05 — SOCIOECONOMIC ANALYSIS
# STEP 1 — LOAD ANALYTICAL DATASET

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np

DATA_PATH = (
    "/content/drive/MyDrive/"
    "NYC_311_2025_BACKUP/"
    "NYC_311_2025_FINAL.parquet"
)

df = pd.read_parquet(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print()
print(df.columns.tolist())


Mounted at /content/drive
Rows: 3655040
Columns: 19

['service_request_id', 'created_at', 'closed_at', 'resolution_hours', 'resolution_days', 'resolution_quality', 'agency', 'complaint_type', 'descriptor', 'status', 'resolution_description', 'incident_zip', 'borough', 'community_board', 'community_board_number', 'community_board_borough', 'latitude', 'longitude', 'geo_quality_final']


## 2. Define ACS Data Requirements



In [2]:
ACS_YEAR = 2024
ACS_DATASET = "acs/acs5"

# New York State FIPS
STATE_FIPS = "36"

# Variables:
# B01003_001E = Total population
# B19013_001E = Median household income
# B17001_001E = Total population for poverty status
# B17001_002E = Population below poverty level
# B23025_003E = Civilian labor force
# B23025_005E = Civilian unemployed population
# B25001_001E = Total housing units

ACS_VARIABLES = [
    "B01003_001E",
    "B19013_001E",
    "B17001_001E",
    "B17001_002E",
    "B23025_003E",
    "B23025_005E",
    "B25001_001E"
]

print("ACS Year:", ACS_YEAR)
print("Dataset:", ACS_DATASET)
print("State FIPS:", STATE_FIPS)
print("Variables:", len(ACS_VARIABLES))
print()
print("Configuration ready.")


ACS Year: 2024
Dataset: acs/acs5
State FIPS: 36
Variables: 7

Configuration ready.


## 3. Download ACS 2024 5-Year Data



In [8]:
# STEP 3 — DOWNLOAD REQUIRED ACS 2024 5-YEAR TRACT DATA

import os
import requests

ACS_DOWNLOAD_DIR = "/content/acs_2024"
os.makedirs(ACS_DOWNLOAD_DIR, exist_ok=True)

BASE_URL = (
    "https://www2.census.gov/programs-surveys/acs/"
    "summary_file/2024/table-based-SF/data/5YRData/"
)

# Required ACS tables
# B01003 = Total Population
# B19013 = Median Household Income
# B17001 = Poverty Status
# B23025 = Employment Status
# B25001 = Housing Units

ACS_TABLES = [
    "b01003",
    "b19013",
    "b17001",
    "b23025",
    "b25001"
]

downloaded_files = []

for table in ACS_TABLES:

    filename = f"acsdt5y2024-{table}.dat"
    url = BASE_URL + filename
    output_path = os.path.join(ACS_DOWNLOAD_DIR, filename)

    print(f"Downloading {filename} ...")

    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    file_size_mb = os.path.getsize(output_path) / (1024 ** 2)

    downloaded_files.append(filename)

    print(f"  Saved: {file_size_mb:.2f} MB")

print()
print("=" * 60)
print("ACS DOWNLOAD COMPLETE")
print("=" * 60)
print("Files downloaded:", len(downloaded_files))
print()
for file in downloaded_files:
    print("✓", file)


  Saved: 17.47 MB
  Saved: 17.09 MB
  Saved: 113.11 MB
  Saved: 38.12 MB
  Saved: 16.97 MB

ACS DOWNLOAD COMPLETE
Files downloaded: 5

✓ acsdt5y2024-b01003.dat
✓ acsdt5y2024-b19013.dat
✓ acsdt5y2024-b17001.dat
✓ acsdt5y2024-b23025.dat
✓ acsdt5y2024-b25001.dat


## 4. Inspect ACS Data Structure



In [9]:
# STEP 4 — INSPECT ACS TABLE FILE STRUCTURE

import os

for filename in downloaded_files:
    path = os.path.join(ACS_DOWNLOAD_DIR, filename)

    print("=" * 80)
    print(filename)
    print("=" * 80)

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for i in range(5):
            line = f.readline()
            print(line[:500].rstrip())

    print()


acsdt5y2024-b01003.dat
GEO_ID|B01003_E001|B01003_M001
0100000US|334922499|-555555555
0100089US|1053712|13108
0100090US|1587|466
0100091US|2631117|2727

acsdt5y2024-b19013.dat
GEO_ID|B19013_E001|B19013_M001
0100000US|80734|166
0100089US|59300|550
0100090US|52188|17624
0100091US|63629|420

acsdt5y2024-b17001.dat
GEO_ID|B17001_E001|B17001_M001|B17001_E002|B17001_M002|B17001_E003|B17001_M003|B17001_E004|B17001_M004|B17001_E005|B17001_M005|B17001_E006|B17001_M006|B17001_E007|B17001_M007|B17001_E008|B17001_M008|B17001_E009|B17001_M009|B17001_E010|B17001_M010|B17001_E011|B17001_M011|B17001_E012|B17001_M012|B17001_E013|B17001_M013|B17001_E014|B17001_M014|B17001_E015|B17001_M015|B17001_E016|B17001_M016|B17001_E017|B17001_M017|B17001_E018|B17001_M018|B17001_E019|B17001_M019|B17001_E020|B17001_M020|B17001_E021|B
0100000US|327079188|15021|40735661|275861|18182378|132107|1627422|24089|331371|6724|2040782|28931|993527|12827|341873|6155|630392|9041|2283344|16917|2038247|14403|1865716|18115|1632983|14

## 5. Extract New York Census Tracts



In [10]:
# STEP 5 — EXTRACT NEW YORK CENSUS TRACTS

import os
import pandas as pd

# New York State FIPS = 36
NY_STATE_FIPS = "36"

acs_tables = {}

for filename in downloaded_files:

    path = os.path.join(ACS_DOWNLOAD_DIR, filename)

    table_name = filename.replace(
        "acsdt5y2024-", ""
    ).replace(".dat", "").upper()

    print(f"Reading {filename} ...")

    table = pd.read_csv(
        path,
        sep="|",
        dtype="string"
    )

    # GEO_ID examples:
    # 1400000US36005000100 = NY census tract
    #
    # The relevant tract GEO_ID begins with:
    # 1400000US + 2-digit state FIPS + 3-digit county FIPS + 6-digit tract

    tract_prefix = "1400000US" + NY_STATE_FIPS

    ny_tracts = table[
        table["GEO_ID"].str.startswith(
            tract_prefix,
            na=False
        )
    ].copy()

    acs_tables[table_name] = ny_tracts

    print(
        f"  Total rows: {len(table):,}"
    )
    print(
        f"  NY tract rows: {len(ny_tracts):,}"
    )
    print(
        f"  Columns: {len(ny_tracts.columns)}"
    )
    print()

print("=" * 70)
print("NEW YORK TRACT EXTRACTION COMPLETE")
print("=" * 70)

for table_name, table in acs_tables.items():
    print(
        f"{table_name}: "
        f"{len(table):,} New York tract records"
    )


Reading acsdt5y2024-b01003.dat ...
  Total rows: 616,690
  NY tract rows: 5,397
  Columns: 3

Reading acsdt5y2024-b19013.dat ...
  Total rows: 543,579
  NY tract rows: 5,397
  Columns: 3

Reading acsdt5y2024-b17001.dat ...
  Total rows: 299,181
  NY tract rows: 5,397
  Columns: 119

Reading acsdt5y2024-b23025.dat ...
  Total rows: 543,579
  NY tract rows: 5,397
  Columns: 15

Reading acsdt5y2024-b25001.dat ...
  Total rows: 616,690
  NY tract rows: 5,397
  Columns: 3

NEW YORK TRACT EXTRACTION COMPLETE
B01003: 5,397 New York tract records
B19013: 5,397 New York tract records
B17001: 5,397 New York tract records
B23025: 5,397 New York tract records
B25001: 5,397 New York tract records


## 6. Build Socioeconomic Tract Dataset



In [11]:
# STEP 6 — BUILD CLEAN ACS SOCIOECONOMIC TRACT DATASET

# 1. Population

population = acs_tables["B01003"][
    ["GEO_ID", "B01003_E001"]
].copy()

population = population.rename(
    columns={
        "B01003_E001": "population"
    }
)

# 2. Median household income

income = acs_tables["B19013"][
    ["GEO_ID", "B19013_E001"]
].copy()

income = income.rename(
    columns={
        "B19013_E001": "median_household_income"
    }
)

# 3. Poverty

poverty = acs_tables["B17001"][
    [
        "GEO_ID",
        "B17001_E001",
        "B17001_E002"
    ]
].copy()

poverty = poverty.rename(
    columns={
        "B17001_E001": "poverty_universe",
        "B17001_E002": "poverty_below"
    }
)

# 4. Labor force / unemployment

employment = acs_tables["B23025"][
    [
        "GEO_ID",
        "B23025_E003",
        "B23025_E005"
    ]
].copy()

employment = employment.rename(
    columns={
        "B23025_E003": "civilian_labor_force",
        "B23025_E005": "unemployed"
    }
)

# 5. Housing units

housing = acs_tables["B25001"][
    ["GEO_ID", "B25001_E001"]
].copy()

housing = housing.rename(
    columns={
        "B25001_E001": "housing_units"
    }
)

# 6. Merge all ACS tables

acs_tract = population.merge(
    income,
    on="GEO_ID",
    how="outer"
)

acs_tract = acs_tract.merge(
    poverty,
    on="GEO_ID",
    how="outer"
)

acs_tract = acs_tract.merge(
    employment,
    on="GEO_ID",
    how="outer"
)

acs_tract = acs_tract.merge(
    housing,
    on="GEO_ID",
    how="outer"
)

# 7. Convert numeric columns

numeric_columns = [
    "population",
    "median_household_income",
    "poverty_universe",
    "poverty_below",
    "civilian_labor_force",
    "unemployed",
    "housing_units"
]

for col in numeric_columns:
    acs_tract[col] = pd.to_numeric(
        acs_tract[col],
        errors="coerce"
    )

# 8. Calculate derived socioeconomic metrics

acs_tract["poverty_rate_pct"] = np.where(
    acs_tract["poverty_universe"] > 0,
    (
        acs_tract["poverty_below"]
        / acs_tract["poverty_universe"]
    ) * 100,
    np.nan
)

acs_tract["unemployment_rate_pct"] = np.where(
    acs_tract["civilian_labor_force"] > 0,
    (
        acs_tract["unemployed"]
        / acs_tract["civilian_labor_force"]
    ) * 100,
    np.nan
)

# 9. Extract Census tract GEOID

acs_tract["tract_geoid"] = (
    acs_tract["GEO_ID"]
    .str.replace("1400000US", "", regex=False)
)

# 10. Final column order

acs_tract = acs_tract[
    [
        "tract_geoid",
        "GEO_ID",
        "population",
        "median_household_income",
        "poverty_universe",
        "poverty_below",
        "poverty_rate_pct",
        "civilian_labor_force",
        "unemployed",
        "unemployment_rate_pct",
        "housing_units"
    ]
].copy()

# 11. Validation

print("=" * 70)
print("ACS SOCIOECONOMIC DATASET")
print("=" * 70)

print("Rows:", len(acs_tract))
print("Columns:", len(acs_tract.columns))
print("Unique tract GEOIDs:", acs_tract["tract_geoid"].nunique())
print()

print("Missing values:")
print(
    acs_tract[
        [
            "population",
            "median_household_income",
            "poverty_rate_pct",
            "unemployment_rate_pct",
            "housing_units"
        ]
    ].isna().sum()
)

print()
print("Population range:")
print(
    acs_tract["population"].min(),
    "to",
    acs_tract["population"].max()
)

print()
print("Median household income range:")
print(
    acs_tract["median_household_income"].min(),
    "to",
    acs_tract["median_household_income"].max()
)

print()
print("Sample:")
display(acs_tract.head())


ACS SOCIOECONOMIC DATASET
Rows: 5397
Columns: 11
Unique tract GEOIDs: 5397

Missing values:
population                   0
median_household_income      0
poverty_rate_pct           135
unemployment_rate_pct      124
housing_units                0
dtype: int64

Population range:
0 to 16963

Median household income range:
-666666666 to 250001

Sample:


,tract_geoid,GEO_ID,population,median_household_income,poverty_universe,poverty_below,poverty_rate_pct,civilian_labor_force,unemployed,unemployment_rate_pct,housing_units
0,36001000100,1400000US36001000100,2294,51551,2294,543,23.670445,973,160,16.443988,1028
1,36001000201,1400000US36001000201,3055,54013,3055,721,23.600655,1497,111,7.414830,1691
2,36001000202,1400000US36001000202,2256,41952,2256,781,34.618794,1405,26,1.850534,1283
3,36001000301,1400000US36001000301,2543,42912,2543,927,36.453008,1124,86,7.651246,1515
4,36001000302,1400000US36001000302,3192,44730,2984,596,19.973190,1742,60,3.444317,2198


## 7. Clean ACS Sentinel Values



In [12]:
# STEP 7 — CLEAN ACS SPECIAL / SENTINEL VALUES

# Census uses negative sentinel values for unavailable /
# not-applicable estimates in some ACS tables.

ACS_SENTINEL = -666666666

# Replace Census sentinel values with proper missing values
acs_tract = acs_tract.replace(
    ACS_SENTINEL,
    np.nan
)

# Recalculate derived rates after sentinel cleaning
acs_tract["poverty_rate_pct"] = np.where(
    acs_tract["poverty_universe"] > 0,
    (
        acs_tract["poverty_below"]
        / acs_tract["poverty_universe"]
    ) * 100,
    np.nan
)

acs_tract["unemployment_rate_pct"] = np.where(
    acs_tract["civilian_labor_force"] > 0,
    (
        acs_tract["unemployed"]
        / acs_tract["civilian_labor_force"]
    ) * 100,
    np.nan
)

# Validation

print("=" * 70)
print("ACS CLEANING VALIDATION")
print("=" * 70)

print("Rows:", len(acs_tract))
print("Unique tract GEOIDs:", acs_tract["tract_geoid"].nunique())
print()

print("Median household income:")
print(
    "Min:",
    acs_tract["median_household_income"].min()
)
print(
    "Max:",
    acs_tract["median_household_income"].max()
)
print(
    "Missing:",
    acs_tract["median_household_income"].isna().sum()
)

print()

print("Population:")
print(
    "Min:",
    acs_tract["population"].min()
)
print(
    "Max:",
    acs_tract["population"].max()
)
print(
    "Missing:",
    acs_tract["population"].isna().sum()
)

print()

print("Poverty rate:")
print(
    "Missing:",
    acs_tract["poverty_rate_pct"].isna().sum()
)
print(
    "Min:",
    acs_tract["poverty_rate_pct"].min()
)
print(
    "Max:",
    acs_tract["poverty_rate_pct"].max()
)

print()

print("Unemployment rate:")
print(
    "Missing:",
    acs_tract["unemployment_rate_pct"].isna().sum()
)
print(
    "Min:",
    acs_tract["unemployment_rate_pct"].min()
)
print(
    "Max:",
    acs_tract["unemployment_rate_pct"].max()
)

print()

print("Remaining Census sentinel values:")
print(
    (acs_tract == ACS_SENTINEL).sum().sum()
)

print()

print("Final socioeconomic columns:")
print(acs_tract.columns.tolist())


ACS CLEANING VALIDATION
Rows: 5397
Unique tract GEOIDs: 5397

Median household income:
Min: 2499
Max: 250001
Missing: 206

Population:
Min: 0
Max: 16963
Missing: 0

Poverty rate:
Missing: 135
Min: 0.0
Max: 100.0

Unemployment rate:
Missing: 124
Min: 0.0
Max: 100.0

Remaining Census sentinel values:
0

Final socioeconomic columns:
['tract_geoid', 'GEO_ID', 'population', 'median_household_income', 'poverty_universe', 'poverty_below', 'poverty_rate_pct', 'civilian_labor_force', 'unemployed', 'unemployment_rate_pct', 'housing_units']


## 9. Prepare Geospatial Tools



In [14]:
# STEP 9 — PREPARE GEOSPATIAL TOOLS

import geopandas as gpd
import pandas as pd
import numpy as np

print("GeoPandas version:", gpd.__version__)
print("Pandas version:", pd.__version__)

print()
print("Geospatial libraries ready.")


GeoPandas version: 1.1.4
Pandas version: 2.2.3

Geospatial libraries ready.


## 10. Download TIGER/Line Census Tracts



In [15]:
# STEP 10 — DOWNLOAD 2024 TIGER/LINE CENSUS TRACT BOUNDARIES

import os
import requests
import zipfile

TIGER_DIR = "/content/tiger_2024"
os.makedirs(TIGER_DIR, exist_ok=True)

TIGER_URL = (
    "https://www2.census.gov/geo/tiger/"
    "TIGER2024/TRACT/tl_2024_36_tract.zip"
)

TIGER_ZIP = os.path.join(
    TIGER_DIR,
    "tl_2024_36_tract.zip"
)

print("Downloading official Census TIGER/Line tract boundaries...")
print(TIGER_URL)
print()

response = requests.get(
    TIGER_URL,
    timeout=300
)

print("HTTP Status:", response.status_code)

response.raise_for_status()

with open(TIGER_ZIP, "wb") as f:
    f.write(response.content)

file_size_mb = os.path.getsize(TIGER_ZIP) / (1024 ** 2)

print(
    f"Downloaded ZIP size: {file_size_mb:.2f} MB"
)

# Extract
with zipfile.ZipFile(TIGER_ZIP, "r") as zip_ref:
    zip_ref.extractall(TIGER_DIR)

print()
print("Extraction complete.")

# Find the shapefile
shapefiles = [
    f for f in os.listdir(TIGER_DIR)
    if f.endswith(".shp")
]

print()
print("Shapefile(s) found:")

for shp in shapefiles:
    print("✓", shp)


https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_36_tract.zip

HTTP Status: 200
Downloaded ZIP size: 8.44 MB

Extraction complete.

Shapefile(s) found:
✓ tl_2024_36_tract.shp


## 11. Validate Census Tract Boundaries



In [16]:
# STEP 11 — LOAD AND VALIDATE CENSUS TRACT BOUNDARIES

TIGER_SHP = (
    "/content/tiger_2024/"
    "tl_2024_36_tract.shp"
)

tracts = gpd.read_file(TIGER_SHP)

print("=" * 70)
print("TIGER/LINE TRACT BOUNDARY VALIDATION")
print("=" * 70)

print("Rows:", len(tracts))
print("Columns:", len(tracts.columns))
print()

print("CRS:", tracts.crs)
print()

print("GEOID column present:", "GEOID" in tracts.columns)
print("Unique GEOIDs:", tracts["GEOID"].nunique())
print()

print("Geometry types:")
print(tracts.geometry.geom_type.value_counts())
print()

print("Sample columns:")
print(tracts.columns.tolist())


TIGER/LINE TRACT BOUNDARY VALIDATION
Rows: 5411
Columns: 14

CRS: EPSG:4269

GEOID column present: True
Unique GEOIDs: 5411

Geometry types:
Polygon         5401
MultiPolygon      10
Name: count, dtype: int64

Sample columns:
['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']


## 12. Reconcile ACS and TIGER/Line GEOIDs



In [17]:
# STEP 12 — RECONCILE ACS TRACTS WITH TIGER/LINE GEOIDS

acs_geoids = set(
    acs_tract["tract_geoid"].astype(str)
)

tiger_geoids = set(
    tracts["GEOID"].astype(str)
)

acs_only = sorted(
    acs_geoids - tiger_geoids
)

tiger_only = sorted(
    tiger_geoids - acs_geoids
)

matched = sorted(
    acs_geoids & tiger_geoids
)

print("=" * 70)
print("ACS ↔ TIGER GEOID RECONCILIATION")
print("=" * 70)

print("ACS tracts:", len(acs_geoids))
print("TIGER tracts:", len(tiger_geoids))
print("Matched GEOIDs:", len(matched))
print("ACS-only GEOIDs:", len(acs_only))
print("TIGER-only GEOIDs:", len(tiger_only))

print()
print("Match rate from ACS:")
print(
    f"{len(matched) / len(acs_geoids) * 100:.2f}%"
)

print()
print("Sample ACS-only GEOIDs:")
print(acs_only[:20])

print()
print("Sample TIGER-only GEOIDs:")
print(tiger_only[:20])


ACS ↔ TIGER GEOID RECONCILIATION
ACS tracts: 5397
TIGER tracts: 5411
Matched GEOIDs: 5397
ACS-only GEOIDs: 0
TIGER-only GEOIDs: 14

Match rate from ACS:
100.00%

Sample ACS-only GEOIDs:
[]

Sample TIGER-only GEOIDs:
['36103122406', '36103122501', '36103145601', '36103145602', '36103145603', '36103145604', '36103145605', '36103145702', '36103146001', '36103146105', '36103146106', '36103146201', '36103146204', '36103201200']


## 13. Prepare NYC 311 Coordinates



In [18]:
# STEP 13 — PREPARE NYC 311 POINTS FOR SPATIAL JOIN

# Keep only records with both latitude and longitude
points_df = df[
    df["latitude"].notna()
    & df["longitude"].notna()
].copy()

print("=" * 70)
print("NYC 311 SPATIAL JOIN INPUT")
print("=" * 70)

print("Total 311 records:", len(df))
print("Records with coordinates:", len(points_df))

print(
    "Coordinate coverage:",
    f"{len(points_df) / len(df) * 100:.2f}%"
)

print()

print("Latitude range:")
print(
    points_df["latitude"].min(),
    "to",
    points_df["latitude"].max()
)

print()

print("Longitude range:")
print(
    points_df["longitude"].min(),
    "to",
    points_df["longitude"].max()
)


NYC 311 SPATIAL JOIN INPUT
Total 311 records: 3655040
Records with coordinates: 3303394
Coordinate coverage: 90.38%

Latitude range:
40.49891293717068 to 40.912868795316655

Longitude range:
-74.25495171973925 to -73.70036545591262


## 14. Create NYC 311 Point GeodataFrame



In [19]:
# STEP 14 — CREATE NYC 311 POINT GEODATAFRAME

points_gdf = gpd.GeoDataFrame(
    points_df,
    geometry=gpd.points_from_xy(
        points_df["longitude"],
        points_df["latitude"]
    ),
    crs="EPSG:4326"
)

print("=" * 70)
print("NYC 311 POINT GEODATAFRAME")
print("=" * 70)

print("Rows:", len(points_gdf))
print("CRS:", points_gdf.crs)
print("Geometry type:")
print(points_gdf.geometry.geom_type.value_counts())

print()
print("Empty geometries:", points_gdf.geometry.is_empty.sum())
print("Missing geometries:", points_gdf.geometry.isna().sum())


NYC 311 POINT GEODATAFRAME
Rows: 3303394
CRS: EPSG:4326
Geometry type:
Point    3303394
Name: count, dtype: int64

Empty geometries: 0
Missing geometries: 0


## 15. Prepare Matched Census Tracts



In [20]:
# STEP 15 — PREPARE ACS-MATCHED CENSUS TRACT BOUNDARIES

# Keep only Census tracts that have ACS socioeconomic data
matched_tracts = tracts[
    tracts["GEOID"].isin(
        acs_tract["tract_geoid"]
    )
].copy()

# Keep only the fields needed for the spatial join
matched_tracts = matched_tracts[
    [
        "GEOID",
        "geometry"
    ]
].copy()

print("=" * 70)
print("MATCHED CENSUS TRACT BOUNDARIES")
print("=" * 70)

print("TIGER boundaries:", len(tracts))
print("ACS-matched boundaries:", len(matched_tracts))
print(
    "Unique GEOIDs:",
    matched_tracts["GEOID"].nunique()
)

print()
print("CRS:", matched_tracts.crs)

print()
print("Geometry types:")
print(
    matched_tracts.geometry.geom_type.value_counts()
)

print()
print(
    "Missing geometries:",
    matched_tracts.geometry.isna().sum()
)


MATCHED CENSUS TRACT BOUNDARIES
TIGER boundaries: 5411
ACS-matched boundaries: 5397
Unique GEOIDs: 5397

CRS: EPSG:4269

Geometry types:
Polygon         5387
MultiPolygon      10
Name: count, dtype: int64

Missing geometries: 0


## 16. Align Coordinate Reference Systems



In [21]:
# STEP 16 — ALIGN CRS FOR SPATIAL JOIN

points_gdf = points_gdf.to_crs(
    matched_tracts.crs
)

print("=" * 70)
print("CRS ALIGNMENT VALIDATION")
print("=" * 70)

print("311 points CRS:", points_gdf.crs)
print("Census tracts CRS:", matched_tracts.crs)

print()
print(
    "CRS match:",
    points_gdf.crs == matched_tracts.crs
)


CRS ALIGNMENT VALIDATION
311 points CRS: EPSG:4269
Census tracts CRS: EPSG:4269

CRS match: True


## 17. Spatial Join: NYC 311 to Census Tracts



In [22]:
# STEP 17 — SPATIAL JOIN NYC 311 → CENSUS TRACTS

print("Starting spatial join...")
print("311 points:", len(points_gdf))
print("Census tracts:", len(matched_tracts))
print()

points_with_tract = gpd.sjoin(
    points_gdf,
    matched_tracts,
    how="left",
    predicate="within"
)

print()
print("=" * 70)
print("SPATIAL JOIN COMPLETE")
print("=" * 70)

print("Input 311 points:", len(points_gdf))
print("Joined records:", len(points_with_tract))

print()
print(
    "Records assigned to Census tract:",
    points_with_tract["GEOID"].notna().sum()
)

print(
    "Records not assigned to Census tract:",
    points_with_tract["GEOID"].isna().sum()
)

print()
print(
    "Tract assignment coverage:",
    f"{points_with_tract['GEOID'].notna().mean() * 100:.2f}%"
)


Starting spatial join...
311 points: 3303394
Census tracts: 5397


SPATIAL JOIN COMPLETE
Input 311 points: 3303394
Joined records: 3303394

Records assigned to Census tract: 3303394
Records not assigned to Census tract: 0

Tract assignment coverage: 100.00%


## 18. Attach Socioeconomic Data



In [23]:
# STEP 18 — ATTACH ACS SOCIOECONOMIC DATA TO 311 RECORDS

# Prepare ACS lookup table
acs_lookup = acs_tract[
    [
        "tract_geoid",
        "population",
        "median_household_income",
        "poverty_rate_pct",
        "unemployment_rate_pct",
        "housing_units"
    ]
].copy()

# Make GEOID types consistent
points_with_tract["GEOID"] = (
    points_with_tract["GEOID"]
    .astype(str)
)

acs_lookup["tract_geoid"] = (
    acs_lookup["tract_geoid"]
    .astype(str)
)

# Merge socioeconomic characteristics
points_with_tract = points_with_tract.merge(
    acs_lookup,
    left_on="GEOID",
    right_on="tract_geoid",
    how="left",
    validate="many_to_one"
)

print("=" * 70)
print("ACS DATA ATTACHMENT VALIDATION")
print("=" * 70)

print("311 records:", len(points_with_tract))
print()

print(
    "Population available:",
    points_with_tract["population"].notna().sum()
)

print(
    "Median income available:",
    points_with_tract["median_household_income"].notna().sum()
)

print(
    "Poverty rate available:",
    points_with_tract["poverty_rate_pct"].notna().sum()
)

print(
    "Unemployment rate available:",
    points_with_tract["unemployment_rate_pct"].notna().sum()
)

print()
print(
    "Unique Census tracts represented:",
    points_with_tract["GEOID"].nunique()
)

print()
print("Sample socioeconomic fields:")
display(
    points_with_tract[
        [
            "service_request_id",
            "GEOID",
            "population",
            "median_household_income",
            "poverty_rate_pct",
            "unemployment_rate_pct",
            "housing_units"
        ]
    ].head()
)


ACS DATA ATTACHMENT VALIDATION
311 records: 3303394

Population available: 3303394
Median income available: 3240877
Poverty rate available: 3267365
Unemployment rate available: 3266459

Unique Census tracts represented: 2334

Sample socioeconomic fields:


,service_request_id,GEOID,population,median_household_income,poverty_rate_pct,unemployment_rate_pct,housing_units
0,63572086,36005046206,3971,67234,1.166920,4.231909,2163
1,63572092,36081046300,4020,75539,12.421738,3.507951,1231
2,63572113,36047119400,4816,84332,35.070598,16.075246,1618
3,63572115,36081017000,2738,108229,5.989774,13.976500,704
4,63572131,36047114600,3130,82228,8.722045,4.672897,895


## 19. Build Tract-Level Analytical Dataset



In [24]:
# STEP 19 — BUILD TRACT-LEVEL 311 + SOCIOECONOMIC DATASET

# Aggregate 311 activity to Census tract
tract_311 = (
    points_with_tract
    .groupby("GEOID")
    .agg(
        total_requests=("service_request_id", "count"),
        valid_resolution_records=("resolution_hours", "count"),
        median_resolution_hours=("resolution_hours", "median"),
        p75_resolution_hours=(
            "resolution_hours",
            lambda x: x.quantile(0.75)
        ),
        p90_resolution_hours=(
            "resolution_hours",
            lambda x: x.quantile(0.90)
        )
    )
    .reset_index()
)

# Resolution coverage within each tract
tract_311["resolution_coverage_pct"] = (
    tract_311["valid_resolution_records"]
    / tract_311["total_requests"]
    * 100
)

# Merge socioeconomic characteristics
tract_analysis = tract_311.merge(
    acs_lookup,
    left_on="GEOID",
    right_on="tract_geoid",
    how="left",
    validate="one_to_one"
)

# Requests per 1,000 residents
tract_analysis["requests_per_1000_residents"] = np.where(
    tract_analysis["population"] > 0,
    (
        tract_analysis["total_requests"]
        / tract_analysis["population"]
        * 1000
    ),
    np.nan
)

# Convert resolution metrics to days
tract_analysis["median_resolution_days"] = (
    tract_analysis["median_resolution_hours"] / 24
)

tract_analysis["p75_resolution_days"] = (
    tract_analysis["p75_resolution_hours"] / 24
)

tract_analysis["p90_resolution_days"] = (
    tract_analysis["p90_resolution_hours"] / 24
)

# Final column order
tract_analysis = tract_analysis[
    [
        "GEOID",
        "total_requests",
        "valid_resolution_records",
        "resolution_coverage_pct",
        "median_resolution_hours",
        "p75_resolution_hours",
        "p90_resolution_hours",
        "median_resolution_days",
        "p75_resolution_days",
        "p90_resolution_days",
        "population",
        "requests_per_1000_residents",
        "median_household_income",
        "poverty_rate_pct",
        "unemployment_rate_pct",
        "housing_units"
    ]
]

print("=" * 70)
print("TRACT-LEVEL SOCIOECONOMIC ANALYSIS DATASET")
print("=" * 70)

print("Rows:", len(tract_analysis))
print("Columns:", len(tract_analysis.columns))

print()
print("Unique Census tracts:", tract_analysis["GEOID"].nunique())

print()
print("Total 311 requests represented:")
print(
    f"{tract_analysis['total_requests'].sum():,}"
)

print()
print("Population available:")
print(
    f"{tract_analysis['population'].notna().sum():,}"
)

print()
print("Requests per 1,000 residents:")
print(
    "Min:",
    tract_analysis["requests_per_1000_residents"].min()
)
print(
    "Median:",
    tract_analysis["requests_per_1000_residents"].median()
)
print(
    "Max:",
    tract_analysis["requests_per_1000_residents"].max()
)

print()
print("Sample:")
display(tract_analysis.head())


TRACT-LEVEL SOCIOECONOMIC ANALYSIS DATASET
Rows: 2334
Columns: 16

Unique Census tracts: 2334

Total 311 requests represented:
3,303,394

Population available:
2,334

Requests per 1,000 residents:
Min: 0.3808556557064873
Median: 328.65907099035934
Max: 46250.0

Sample:


,GEOID,total_requests,valid_resolution_records,resolution_coverage_pct,median_resolution_hours,p75_resolution_hours,p90_resolution_hours,median_resolution_days,p75_resolution_days,p90_resolution_days,population,requests_per_1000_residents,median_household_income,poverty_rate_pct,unemployment_rate_pct,housing_units
0,36005000100,35,34,97.142857,19.950417,197.075069,1440.828778,0.831267,8.211461,60.034532,2950,11.864407,<NA>,NaN,NaN,0
1,36005000200,1572,1538,97.837150,6.499861,28.702917,183.867222,0.270828,1.195955,7.661134,5061,310.610551,123729,12.882829,6.098032,1484
2,36005000400,1510,1478,97.880795,7.516667,42.725000,257.291500,0.313194,1.780208,10.720479,6649,227.101820,105924,4.466837,8.423586,2255
3,36005001600,1490,1459,97.919463,21.059722,119.256944,608.291167,0.877488,4.969039,25.345465,5611,265.549813,52147,15.162319,8.852105,2142
4,36005001901,1860,1831,98.440860,23.338333,140.746667,836.855000,0.972431,5.864444,34.868958,2260,823.008850,58083,22.453499,10.585745,1160


## 20. Validate Tract-Level Dataset



In [25]:
# STEP 20 — VALIDATE TRACT-LEVEL ANALYTICAL DATASET

print("=" * 70)
print("TRACT-LEVEL DATA QUALITY VALIDATION")
print("=" * 70)

# 1. Row and GEOID uniqueness
print("Rows:", len(tract_analysis))
print("Unique GEOIDs:", tract_analysis["GEOID"].nunique())
print(
    "Duplicate GEOIDs:",
    tract_analysis["GEOID"].duplicated().sum()
)

# 2. Request reconciliation
print()
print("311 requests in tract dataset:")
print(
    f"{tract_analysis['total_requests'].sum():,}"
)

print(
    "Expected geocoded 311 requests:",
    f"{len(points_with_tract):,}"
)

print(
    "Request reconciliation:",
    tract_analysis["total_requests"].sum()
    == len(points_with_tract)
)

# 3. Socioeconomic coverage
print()
print("Socioeconomic data coverage:")

for col in [
    "population",
    "median_household_income",
    "poverty_rate_pct",
    "unemployment_rate_pct",
    "housing_units"
]:
    available = tract_analysis[col].notna().sum()
    coverage = available / len(tract_analysis) * 100

    print(
        f"{col}: "
        f"{available:,} / {len(tract_analysis):,} "
        f"({coverage:.2f}%)"
    )

# 4. Resolution validation
print()
print("Resolution validation:")

print(
    "Negative median resolution:",
    (
        tract_analysis["median_resolution_hours"] < 0
    ).sum()
)

print(
    "Negative P75 resolution:",
    (
        tract_analysis["p75_resolution_hours"] < 0
    ).sum()
)

print(
    "Negative P90 resolution:",
    (
        tract_analysis["p90_resolution_hours"] < 0
    ).sum()
)

print(
    "P75 < median:",
    (
        tract_analysis["p75_resolution_hours"]
        < tract_analysis["median_resolution_hours"]
    ).sum()
)

print(
    "P90 < P75:",
    (
        tract_analysis["p90_resolution_hours"]
        < tract_analysis["p75_resolution_hours"]
    ).sum()
)

# 5. Rate validation
print()
print("Population-adjusted request rate validation:")

print(
    "Zero/negative population tracts:",
    (
        tract_analysis["population"] <= 0
    ).sum()
)

print(
    "Missing request rates:",
    tract_analysis[
        "requests_per_1000_residents"
    ].isna().sum()
)

# 6. Final status
checks = {
    "Unique GEOIDs":
        tract_analysis["GEOID"].nunique()
        == len(tract_analysis),

    "Request reconciliation":
        tract_analysis["total_requests"].sum()
        == len(points_with_tract),

    "No negative median":
        (
            tract_analysis["median_resolution_hours"]
            < 0
        ).sum() == 0,

    "No negative P75":
        (
            tract_analysis["p75_resolution_hours"]
            < 0
        ).sum() == 0,

    "No negative P90":
        (
            tract_analysis["p90_resolution_hours"]
            < 0
        ).sum() == 0,

    "P75 >= median":
        (
            tract_analysis["p75_resolution_hours"]
            >= tract_analysis["median_resolution_hours"]
        ).all(),

    "P90 >= P75":
        (
            tract_analysis["p90_resolution_hours"]
            >= tract_analysis["p75_resolution_hours"]
        ).all()
}

print()
print("=" * 70)
print("VALIDATION STATUS")
print("=" * 70)

for check, result in checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print()
print(
    "Overall:",
    "PASS" if all(checks.values()) else "REVIEW REQUIRED"
)


TRACT-LEVEL DATA QUALITY VALIDATION
Rows: 2334
Unique GEOIDs: 2334
Duplicate GEOIDs: 0

311 requests in tract dataset:
3,303,394
Expected geocoded 311 requests: 3,303,394
Request reconciliation: True

Socioeconomic data coverage:
population: 2,334 / 2,334 (100.00%)
median_household_income: 2,211 / 2,334 (94.73%)
poverty_rate_pct: 2,251 / 2,334 (96.44%)
unemployment_rate_pct: 2,250 / 2,334 (96.40%)
housing_units: 2,334 / 2,334 (100.00%)

Resolution validation:
Negative median resolution: 0
Negative P75 resolution: 0
Negative P90 resolution: 0
P75 < median: 0
P90 < P75: 0

Population-adjusted request rate validation:
Zero/negative population tracts: 79
Missing request rates: 79

VALIDATION STATUS
Unique GEOIDs: PASS
Request reconciliation: PASS
No negative median: PASS
No negative P75: PASS
No negative P90: PASS
P75 >= median: PASS
P90 >= P75: PASS

Overall: PASS


## 21. Socioeconomic Associations



In [26]:
# STEP 21 — SOCIOECONOMIC ASSOCIATION ANALYSIS

from scipy.stats import pearsonr, spearmanr

analysis_pairs = [
    (
        "requests_per_1000_residents",
        "median_household_income",
        "Request rate vs median household income"
    ),
    (
        "requests_per_1000_residents",
        "poverty_rate_pct",
        "Request rate vs poverty rate"
    ),
    (
        "requests_per_1000_residents",
        "unemployment_rate_pct",
        "Request rate vs unemployment rate"
    ),
    (
        "median_resolution_hours",
        "median_household_income",
        "Median resolution vs median household income"
    ),
    (
        "median_resolution_hours",
        "poverty_rate_pct",
        "Median resolution vs poverty rate"
    ),
    (
        "median_resolution_hours",
        "unemployment_rate_pct",
        "Median resolution vs unemployment rate"
    )
]

results = []

for x_col, y_col, relationship in analysis_pairs:

    pair = tract_analysis[
        [x_col, y_col]
    ].dropna()

    if len(pair) >= 3:

        pearson_r, pearson_p = pearsonr(
            pair[x_col],
            pair[y_col]
        )

        spearman_rho, spearman_p = spearmanr(
            pair[x_col],
            pair[y_col]
        )

        results.append({
            "relationship": relationship,
            "n_tracts": len(pair),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p
        })

correlation_results = pd.DataFrame(results)

print("=" * 70)
print("SOCIOECONOMIC ASSOCIATION ANALYSIS")
print("=" * 70)

display(
    correlation_results.round(4)
)


SOCIOECONOMIC ASSOCIATION ANALYSIS


,relationship,n_tracts,pearson_r,pearson_p,spearman_rho,spearman_p
0,Request rate vs median household income,2211,-0.0011,0.9601,-0.1181,0.0
1,Request rate vs poverty rate,2251,0.0928,0.0000,0.1636,0.0
2,Request rate vs unemployment rate,2250,0.1168,0.0000,0.1083,0.0
3,Median resolution vs median household income,2211,-0.0006,0.9783,-0.1319,0.0
4,Median resolution vs poverty rate,2251,-0.0118,0.5760,0.1182,0.0
5,Median resolution vs unemployment rate,2250,-0.0120,0.5708,0.0938,0.0


## 22. Spearman Correlation Analysis



In [27]:
# STEP 22 — SPEARMAN CORRELATION MATRIX

corr_columns = [
    "requests_per_1000_residents",
    "median_household_income",
    "poverty_rate_pct",
    "unemployment_rate_pct",
    "median_resolution_days"
]

corr_labels = {
    "requests_per_1000_residents": "311 Requests per 1,000",
    "median_household_income": "Median Household Income",
    "poverty_rate_pct": "Poverty Rate",
    "unemployment_rate_pct": "Unemployment Rate",
    "median_resolution_days": "Median Resolution (Days)"
}

spearman_matrix = (
    tract_analysis[corr_columns]
    .rename(columns=corr_labels)
    .corr(method="spearman")
)

print("=" * 70)
print("SPEARMAN CORRELATION MATRIX")
print("=" * 70)

display(
    spearman_matrix.round(3)
)


SPEARMAN CORRELATION MATRIX


,"311 Requests per 1,000",Median Household Income,Poverty Rate,Unemployment Rate,Median Resolution (Days)
"311 Requests per 1,000",1.000,-0.118,0.164,0.108,-0.061
Median Household Income,-0.118,1.000,-0.741,-0.417,-0.132
Poverty Rate,0.164,-0.741,1.000,0.427,0.118
Unemployment Rate,0.108,-0.417,0.427,1.000,0.094
Median Resolution (Days),-0.061,-0.132,0.118,0.094,1.000


## 23. Poverty Quartile Analysis



In [28]:
# STEP 23 — POVERTY QUARTILE SERVICE PROFILE

poverty_df = tract_analysis[
    [
        "GEOID",
        "total_requests",
        "requests_per_1000_residents",
        "median_resolution_days",
        "p75_resolution_days",
        "poverty_rate_pct"
    ]
].dropna(
    subset=["poverty_rate_pct"]
).copy()

# Create four equal-sized tract groups by poverty rate
poverty_df["poverty_quartile"] = pd.qcut(
    poverty_df["poverty_rate_pct"],
    q=4,
    labels=[
        "Q1 - Lower Poverty",
        "Q2",
        "Q3",
        "Q4 - Higher Poverty"
    ],
    duplicates="drop"
)

poverty_quartile_summary = (
    poverty_df
    .groupby("poverty_quartile", observed=False)
    .agg(
        tracts=("GEOID", "count"),
        total_requests=("total_requests", "sum"),
        median_requests_per_1000=(
            "requests_per_1000_residents",
            "median"
        ),
        median_resolution_days=(
            "median_resolution_days",
            "median"
        ),
        median_p75_resolution_days=(
            "p75_resolution_days",
            "median"
        ),
        median_poverty_rate=(
            "poverty_rate_pct",
            "median"
        )
    )
    .reset_index()
)

print("=" * 70)
print("POVERTY QUARTILE SERVICE PROFILE")
print("=" * 70)

display(
    poverty_quartile_summary.round(3)
)


POVERTY QUARTILE SERVICE PROFILE


,poverty_quartile,tracts,total_requests,median_requests_per_1000,median_resolution_days,median_p75_resolution_days,median_poverty_rate
0,Q1 - Lower Poverty,563,643793,288.717,0.279,2.894,5.188
1,Q2,563,728891,320.977,0.295,2.714,10.945
2,Q3,562,833237,349.751,0.360,2.885,17.871
3,Q4 - Higher Poverty,563,1061444,375.168,0.542,3.357,32.314


## 24. Income Quartile Analysis



In [29]:
# STEP 24 — INCOME QUARTILE SERVICE PROFILE

income_df = tract_analysis[
    [
        "GEOID",
        "total_requests",
        "requests_per_1000_residents",
        "median_resolution_days",
        "p75_resolution_days",
        "median_household_income"
    ]
].dropna(
    subset=["median_household_income"]
).copy()

# Exclude ACS top-coded income value from quartile construction
income_df = income_df[
    income_df["median_household_income"] < 250001
].copy()

# Create four equal-sized tract groups by household income
income_df["income_quartile"] = pd.qcut(
    income_df["median_household_income"],
    q=4,
    labels=[
        "Q1 - Lower Income",
        "Q2",
        "Q3",
        "Q4 - Higher Income"
    ],
    duplicates="drop"
)

income_quartile_summary = (
    income_df
    .groupby("income_quartile", observed=False)
    .agg(
        tracts=("GEOID", "count"),
        total_requests=("total_requests", "sum"),
        median_requests_per_1000=(
            "requests_per_1000_residents",
            "median"
        ),
        median_resolution_days=(
            "median_resolution_days",
            "median"
        ),
        median_p75_resolution_days=(
            "p75_resolution_days",
            "median"
        ),
        median_household_income=(
            "median_household_income",
            "median"
        )
    )
    .reset_index()
)

print("=" * 70)
print("INCOME QUARTILE SERVICE PROFILE")
print("=" * 70)

display(
    income_quartile_summary.round(3)
)


INCOME QUARTILE SERVICE PROFILE


,income_quartile,tracts,total_requests,median_requests_per_1000,median_resolution_days,median_p75_resolution_days,median_household_income
0,Q1 - Lower Income,550,1071242,374.936,0.610,3.423,45285.5
1,Q2,550,724196,337.563,0.311,2.584,72291.0
2,Q3,548,691923,316.014,0.274,2.569,93643.5
3,Q4 - Higher Income,550,726185,296.080,0.326,3.145,134525.0


## 25. Request Rate Distribution



In [30]:
# STEP 25 — REQUEST RATE DISTRIBUTION AND EXTREME VALUES

rate_df = tract_analysis[
    [
        "GEOID",
        "total_requests",
        "population",
        "requests_per_1000_residents"
    ]
].dropna(
    subset=["requests_per_1000_residents"]
).copy()

print("=" * 70)
print("REQUEST RATE DISTRIBUTION")
print("=" * 70)

print(
    rate_df["requests_per_1000_residents"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(2)
)

print()
print("=" * 70)
print("HIGHEST REQUEST-RATE TRACTS")
print("=" * 70)

display(
    rate_df
    .sort_values(
        "requests_per_1000_residents",
        ascending=False
    )
    .head(15)
    .round(2)
)


REQUEST RATE DISTRIBUTION
count     2255.00
mean       518.27
std       1895.49
min          0.38
1%          52.02
5%         137.68
10%        174.06
25%        242.68
50%        328.66
75%        469.28
90%        635.38
95%        829.43
99%       2677.58
max      46250.00
Name: requests_per_1000_residents, dtype: float64

HIGHEST REQUEST-RATE TRACTS


,GEOID,total_requests,population,requests_per_1000_residents
1894,36081056100,555,12,46250.00
1472,36061029700,508,13,39076.92
373,36047001803,422,11,38363.64
1291,36061011300,3706,136,27250.00
1267,36061009400,1889,70,26985.71
321,36005042000,121184,4660,26005.15
244,36005033400,536,32,16750.00
2062,36081091800,90,8,11250.00
1626,36081017902,723,69,10478.26
402,36047005302,1184,123,9626.02


## 26. Population Distribution



In [31]:
# STEP 26 — CENSUS TRACT POPULATION DISTRIBUTION

population_df = tract_analysis[
    [
        "GEOID",
        "population",
        "total_requests",
        "requests_per_1000_residents"
    ]
].copy()

print("=" * 70)
print("CENSUS TRACT POPULATION DISTRIBUTION")
print("=" * 70)

print(
    population_df["population"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(2)
)

print()
print("=" * 70)
print("POPULATION THRESHOLD COUNTS")
print("=" * 70)

thresholds = [
    0,
    100,
    250,
    500,
    1000,
    2500,
    5000
]

for threshold in thresholds:

    count = (
        population_df["population"] >= threshold
    ).sum()

    print(
        f"Population >= {threshold:,}: "
        f"{count:,} tracts "
        f"({count / len(population_df) * 100:.2f}%)"
    )


CENSUS TRACT POPULATION DISTRIBUTION
count     2334.0
mean     3655.58
std      2016.27
min          0.0
1%           0.0
5%        893.85
10%       1490.9
25%      2237.25
50%       3423.5
75%       4777.0
90%       6266.8
95%       7243.9
99%      9658.08
max      16963.0
Name: population, dtype: Float64

POPULATION THRESHOLD COUNTS
Population >= 0: 2,334 tracts (100.00%)
Population >= 100: 2,240 tracts (95.97%)
Population >= 250: 2,233 tracts (95.67%)
Population >= 500: 2,223 tracts (95.24%)
Population >= 1,000: 2,207 tracts (94.56%)
Population >= 2,500: 1,635 tracts (70.05%)
Population >= 5,000: 525 tracts (22.49%)


## 27. Population Threshold Sensitivity



In [32]:
# STEP 27 — POPULATION-THRESHOLD SENSITIVITY ANALYSIS

sensitivity_df = tract_analysis[
    tract_analysis["population"] >= 1000
].copy()

print("=" * 70)
print("POPULATION-THRESHOLD SENSITIVITY ANALYSIS")
print("=" * 70)

print(
    "Tracts included:",
    len(sensitivity_df)
)

print(
    "Share of tracts:",
    f"{len(sensitivity_df) / len(tract_analysis) * 100:.2f}%"
)

print()

# Recalculate correlations on the thresholded dataset
sensitivity_pairs = [
    (
        "requests_per_1000_residents",
        "median_household_income",
        "Request rate vs median household income"
    ),
    (
        "requests_per_1000_residents",
        "poverty_rate_pct",
        "Request rate vs poverty rate"
    ),
    (
        "requests_per_1000_residents",
        "unemployment_rate_pct",
        "Request rate vs unemployment rate"
    ),
    (
        "median_resolution_days",
        "median_household_income",
        "Median resolution vs median household income"
    ),
    (
        "median_resolution_days",
        "poverty_rate_pct",
        "Median resolution vs poverty rate"
    ),
    (
        "median_resolution_days",
        "unemployment_rate_pct",
        "Median resolution vs unemployment rate"
    )
]

sensitivity_results = []

for x_col, y_col, relationship in sensitivity_pairs:

    pair = sensitivity_df[
        [x_col, y_col]
    ].dropna()

    if len(pair) >= 3:

        pearson_r, pearson_p = pearsonr(
            pair[x_col],
            pair[y_col]
        )

        spearman_rho, spearman_p = spearmanr(
            pair[x_col],
            pair[y_col]
        )

        sensitivity_results.append({
            "relationship": relationship,
            "n_tracts": len(pair),
            "pearson_r": pearson_r,
            "spearman_rho": spearman_rho
        })

sensitivity_results = pd.DataFrame(
    sensitivity_results
)

print("=" * 70)
print("THRESHOLDED CORRELATION RESULTS")
print("=" * 70)

display(
    sensitivity_results.round(4)
)

print()
print("=" * 70)
print("THRESHOLDED REQUEST-RATE DISTRIBUTION")
print("=" * 70)

print(
    sensitivity_df[
        "requests_per_1000_residents"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    ).round(2)
)


POPULATION-THRESHOLD SENSITIVITY ANALYSIS
Tracts included: 2207
Share of tracts: 94.56%

THRESHOLDED CORRELATION RESULTS


,relationship,n_tracts,pearson_r,spearman_rho
0,Request rate vs median household income,2189,-0.0260,-0.1263
1,Request rate vs poverty rate,2205,0.0758,0.1707
2,Request rate vs unemployment rate,2205,0.0967,0.1241
3,Median resolution vs median household income,2189,-0.0001,-0.1318
4,Median resolution vs poverty rate,2205,-0.0116,0.1232
5,Median resolution vs unemployment rate,2205,-0.0124,0.0933



THRESHOLDED REQUEST-RATE DISTRIBUTION
count     2207.00
mean       391.26
std        605.79
min          0.38
1%          51.90
5%         138.66
25%        240.74
50%        326.15
75%        459.27
90%        612.94
95%        746.85
99%       1267.37
max      26005.15
Name: requests_per_1000_residents, dtype: float64


## 28. Final Tract-Level Dataset



In [33]:
# STEP 28 — FINAL TRACT-LEVEL SOCIOECONOMIC DATASET

tract_analysis_final = tract_analysis.copy()

# Analytical eligibility flag for population-adjusted comparisons
tract_analysis_final["population_threshold_1000"] = (
    tract_analysis_final["population"] >= 1000
)

# Socioeconomic analysis eligibility
tract_analysis_final["socioeconomic_analysis_eligible"] = (
    tract_analysis_final["population_threshold_1000"]
    & tract_analysis_final["median_household_income"].notna()
    & tract_analysis_final["poverty_rate_pct"].notna()
    & tract_analysis_final["unemployment_rate_pct"].notna()
)

print("=" * 70)
print("FINAL TRACT-LEVEL SOCIOECONOMIC DATASET")
print("=" * 70)

print("Rows:", len(tract_analysis_final))
print("Columns:", len(tract_analysis_final.columns))

print()
print(
    "Population >= 1,000:",
    tract_analysis_final[
        "population_threshold_1000"
    ].sum()
)

print(
    "Socioeconomic analysis eligible:",
    tract_analysis_final[
        "socioeconomic_analysis_eligible"
    ].sum()
)

print()
print("Columns:")
for i, col in enumerate(
    tract_analysis_final.columns,
    start=1
):
    print(f"{i}. {col}")

print()
print("Validation:")
print(
    "Unique GEOIDs:",
    tract_analysis_final["GEOID"].nunique()
)

print(
    "Duplicate GEOIDs:",
    tract_analysis_final["GEOID"].duplicated().sum()
)


FINAL TRACT-LEVEL SOCIOECONOMIC DATASET
Rows: 2334
Columns: 18

Population >= 1,000: 2207
Socioeconomic analysis eligible: 2189

Columns:
1. GEOID
2. total_requests
3. valid_resolution_records
4. resolution_coverage_pct
5. median_resolution_hours
6. p75_resolution_hours
7. p90_resolution_hours
8. median_resolution_days
9. p75_resolution_days
10. p90_resolution_days
11. population
12. requests_per_1000_residents
13. median_household_income
14. poverty_rate_pct
15. unemployment_rate_pct
16. housing_units
17. population_threshold_1000
18. socioeconomic_analysis_eligible

Validation:
Unique GEOIDs: 2334
Duplicate GEOIDs: 0


## 29. Complaint Mix by Poverty Quartile



In [34]:
# STEP 29 — MAJOR COMPLAINT MIX BY POVERTY QUARTILE

TOP_COMPLAINTS = [
    "Illegal Parking",
    "Noise - Residential",
    "HEAT/HOT WATER",
    "Blocked Driveway",
    "Noise - Street/Sidewalk"
]

# Start from coordinate-valid 311 records
complaint_socio = points_with_tract[
    [
        "service_request_id",
        "GEOID",
        "complaint_type"
    ]
].copy()

# Attach poverty rate
complaint_socio = complaint_socio.merge(
    acs_lookup[
        [
            "tract_geoid",
            "poverty_rate_pct"
        ]
    ],
    left_on="GEOID",
    right_on="tract_geoid",
    how="left",
    validate="many_to_one"
)

# Keep tracts with valid poverty data
complaint_socio = complaint_socio[
    complaint_socio["poverty_rate_pct"].notna()
].copy()

# Create poverty quartiles using tract-level poverty rate
poverty_lookup = (
    tract_analysis[
        [
            "GEOID",
            "poverty_rate_pct"
        ]
    ]
    .dropna(subset=["poverty_rate_pct"])
    .copy()
)

poverty_lookup["poverty_quartile"] = pd.qcut(
    poverty_lookup["poverty_rate_pct"],
    q=4,
    labels=[
        "Q1 - Lower Poverty",
        "Q2",
        "Q3",
        "Q4 - Higher Poverty"
    ],
    duplicates="drop"
)

# Attach quartile to each 311 record
complaint_socio = complaint_socio.merge(
    poverty_lookup[
        [
            "GEOID",
            "poverty_quartile"
        ]
    ],
    on="GEOID",
    how="left",
    validate="many_to_one"
)

# Keep only the five major complaint categories
complaint_socio = complaint_socio[
    complaint_socio["complaint_type"].isin(
        TOP_COMPLAINTS
    )
].copy()

# Count requests by poverty quartile and complaint type
complaint_mix = (
    complaint_socio
    .groupby(
        [
            "poverty_quartile",
            "complaint_type"
        ],
        observed=False
    )
    .size()
    .reset_index(
        name="requests"
    )
)

# Calculate share within each poverty quartile
quartile_totals = (
    complaint_socio
    .groupby(
        "poverty_quartile",
        observed=False
    )
    .size()
    .reset_index(
        name="top5_total_requests"
    )
)

complaint_mix = complaint_mix.merge(
    quartile_totals,
    on="poverty_quartile",
    how="left"
)

complaint_mix["share_of_top5_pct"] = (
    complaint_mix["requests"]
    / complaint_mix["top5_total_requests"]
    * 100
)

print("=" * 70)
print("MAJOR COMPLAINT MIX BY POVERTY QUARTILE")
print("=" * 70)

display(
    complaint_mix.sort_values(
        [
            "poverty_quartile",
            "requests"
        ],
        ascending=[True, False]
    ).round(2)
)


MAJOR COMPLAINT MIX BY POVERTY QUARTILE


,poverty_quartile,complaint_type,requests,top5_total_requests,share_of_top5_pct
2,Q1 - Lower Poverty,Illegal Parking,142904,262733,54.39
3,Q1 - Lower Poverty,Noise - Residential,38948,262733,14.82
1,Q1 - Lower Poverty,HEAT/HOT WATER,35231,262733,13.41
0,Q1 - Lower Poverty,Blocked Driveway,28734,262733,10.94
4,Q1 - Lower Poverty,Noise - Street/Sidewalk,16916,262733,6.44
7,Q2,Illegal Parking,133891,311192,43.03
8,Q2,Noise - Residential,62757,311192,20.17
6,Q2,HEAT/HOT WATER,48294,311192,15.52
5,Q2,Blocked Driveway,43726,311192,14.05
9,Q2,Noise - Street/Sidewalk,22524,311192,7.24


## 30. Socioeconomic Findings Summary



In [35]:
# STEP 30 — FINAL SOCIOECONOMIC FINDINGS SUMMARY

findings = {
    "Total tract records": len(tract_analysis_final),
    "Population >= 1,000 tracts": int(
        tract_analysis_final[
            "population_threshold_1000"
        ].sum()
    ),
    "Socioeconomic analysis eligible tracts": int(
        tract_analysis_final[
            "socioeconomic_analysis_eligible"
        ].sum()
    ),
    "Geocoded 311 requests analyzed": int(
        tract_analysis_final[
            "total_requests"
        ].sum()
    ),
    "Median 311 requests per 1,000 residents": (
        sensitivity_df[
            "requests_per_1000_residents"
        ].median()
    ),
    "Median poverty rate (%)": (
        poverty_df[
            "poverty_rate_pct"
        ].median()
    ),
    "Median household income ($)": (
        income_df[
            "median_household_income"
        ].median()
    ),
    "Median tract resolution (days)": (
        tract_analysis_final[
            "median_resolution_days"
        ].median()
    )
}

findings_df = pd.DataFrame(
    list(findings.items()),
    columns=[
        "metric",
        "value"
    ]
)

print("=" * 70)
print("FINAL SOCIOECONOMIC FINDINGS SUMMARY")
print("=" * 70)

display(
    findings_df
)


FINAL SOCIOECONOMIC FINDINGS SUMMARY


,metric,value
0,Total tract records,2.334000e+03
1,"Population >= 1,000 tracts",2.207000e+03
2,Socioeconomic analysis eligible tracts,2.189000e+03
3,Geocoded 311 requests analyzed,3.303394e+06
4,"Median 311 requests per 1,000 residents",3.261503e+02
5,Median poverty rate (%),1.414119e+01
6,Median household income ($),8.250000e+04
7,Median tract resolution (days),3.470747e-01


## 31. Final Quality Validation



In [36]:
# STEP 31 — FINAL NOTEBOOK 05 QUALITY VALIDATION

final_checks = {
    "Expected tract dataset exists":
        len(tract_analysis_final) > 0,

    "Unique GEOIDs":
        tract_analysis_final["GEOID"].nunique()
        == len(tract_analysis_final),

    "No duplicate GEOIDs":
        tract_analysis_final["GEOID"].duplicated().sum() == 0,

    "311 request reconciliation":
        tract_analysis_final["total_requests"].sum()
        == len(points_with_tract),

    "Population available":
        tract_analysis_final["population"].notna().all(),

    "No negative median resolution":
        (
            tract_analysis_final[
                "median_resolution_hours"
            ] < 0
        ).sum() == 0,

    "No negative P75 resolution":
        (
            tract_analysis_final[
                "p75_resolution_hours"
            ] < 0
        ).sum() == 0,

    "No negative P90 resolution":
        (
            tract_analysis_final[
                "p90_resolution_hours"
            ] < 0
        ).sum() == 0,

    "P75 >= median":
        (
            tract_analysis_final[
                "p75_resolution_hours"
            ]
            >=
            tract_analysis_final[
                "median_resolution_hours"
            ]
        ).all(),

    "P90 >= P75":
        (
            tract_analysis_final[
                "p90_resolution_hours"
            ]
            >=
            tract_analysis_final[
                "p75_resolution_hours"
            ]
        ).all(),

    "No invalid population-adjusted rates":
        (
            tract_analysis_final.loc[
                tract_analysis_final["population"] > 0,
                "requests_per_1000_residents"
            ].notna()
        ).all(),

    "Socioeconomic eligible count valid":
        tract_analysis_final[
            "socioeconomic_analysis_eligible"
        ].sum() == 2189
}

print("=" * 70)
print("FINAL NOTEBOOK 05 QUALITY VALIDATION")
print("=" * 70)

for check, result in final_checks.items():
    print(
        f"{check}:",
        "PASS" if result else "FAIL"
    )

print()
print("=" * 70)

overall_status = all(final_checks.values())

print(
    "OVERALL STATUS:",
    "PASS" if overall_status else "REVIEW REQUIRED"
)


FINAL NOTEBOOK 05 QUALITY VALIDATION
Expected tract dataset exists: PASS
Unique GEOIDs: PASS
No duplicate GEOIDs: PASS
311 request reconciliation: PASS
Population available: PASS
No negative median resolution: PASS
No negative P75 resolution: PASS
No negative P90 resolution: PASS
P75 >= median: PASS
P90 >= P75: PASS
No invalid population-adjusted rates: PASS
Socioeconomic eligible count valid: PASS

OVERALL STATUS: PASS


## Conclusion

The socioeconomic analysis combines tract-level service request patterns with ACS socioeconomic indicators. Results are interpreted as observed associations at the Census tract level and are not treated as causal relationships.
